# s1 | Odor-plume tracking with a PPO-trained RNN (Singh et al. 2023)

This notebook trains a recurrent agent with **reinforcement learning** to track
a turbulent odor plume to its source, and reproduces the behavioral and
neural-dynamical analyses of

> Singh, van Breugel, Rao & Brunton (2023), *Emergent behaviour and neural
> dynamics in artificial agents tracking odour plumes*, Nature Machine
> Intelligence.

**Why this problem is interesting.** Insects find mates and food by following
odor plumes through moving air. This looks like it should be easy (smell the
odor, walk towards more of it) but turbulence destroys the smooth gradient
that chemotaxis needs. A real plume is a train of discrete puffs separated by
long blanks, and it meanders as the wind shifts, so the instantaneous odor
signal says almost nothing about where the source is. The agent observes only
its local wind and a binary-ish whiff signal; the information it needs lives
in the *history* of those observations. Formally this is a partially
observable control problem, and it is exactly the setting where a recurrent
network should have something a feedforward reflex does not.

**The paper's question.** If you train an RNN end-to-end with PPO, telling it
only that reaching the source is rewarding, what behavior and what internal
dynamics emerge? The paper's answers, which this notebook sets out to test:

1. Behavior decomposes into three regimes (TRACK / RECOVER / SEARCH) defined
   by the time since the last odor encounter, echoing the surge / cast /
   search modules of insect navigation.
2. When the wind rotates mid-flight, the agent keeps following the **plume's
   local centerline** rather than the current wind, a testable prediction for
   insect experiments.
3. The hidden state encodes four memory variables that were never given as
   inputs: head direction, time since last odor, a short-window odor average
   (~0.3 s) and a longer-window encounter rate (~1.9 s).
4. The population dynamics are low-dimensional and split into two dynamical
   regimes: a funnel-like tracking manifold and a limit-cycle-like search
   oscillation.
5. Training reorganizes the recurrent eigenspectrum (a few unstable modes
   appear) and separates integration timescales into fast dynamics plus one
   or two slow memory modes.

**How this maps onto the NeuralRNN layers.** Compared with the other two RL
notebooks: notebook 18 is discrete-action PPO with a Dale-constrained CTRNN;
notebook 19 is critic-only TD learning on an offline dataset. Here the action
space is continuous, the environment replays a stored turbulence simulation,
and the analysis suite is the dynamical-systems core of the paper.

| layer | this notebook |
|---|---|
| data | `rl.envs.plume` (`PlumeEnv`), replaying plume simulations generated by `plume_sim` |
| model | `ActorCriticModel` with a CTRNN-tanh core (alpha = 1: an exact vanilla RNN) and continuous Gaussian actor/critic MLP heads, built by `build_plume_agent("vrnn")` |
| loss | `PPOLoss` (clipped surrogate + value loss + entropy bonus) |
| trainer | `RLTrainer` (recurrent PPO with GAE), behind `VecNormalize` as in the reference |
| analysis | notebook-local utilities ported from the paper's `log_analysis.py` (regimes, course direction, odor-memory features, common subspace, eigenspectra, transition times) |

**Scale and cost.** The paper trains 14 seeds and reports the top 5; we train
5 seeds with the paper's exact step budget (1M + 4M environment steps per
seed, about 10 h on CPU in total). Everything is load-first: the checkpoints
and evaluation logs under `models/s1/` are reused when present, so the
notebook runs end-to-end in a few minutes. Delete a checkpoint folder to
retrain that stage.

**Contents**

1. Setup
2. The plume simulation and the environment
3. The agent: a vanilla tanh RNN actor-critic
4. Two-stage PPO training
5. Fixed-grid evaluation and seed selection
6. Success across wind regimes and plume sparsity
7. Example trajectories
8. Behavioral regimes: TRACK / RECOVER / SEARCH
9. Agents track the plume centerline, not the wind
10. What the RNN remembers
11. Common subspace and limit-cycle search dynamics
12. Eigenspectrum and integration timescales
13. Transition-time asymmetry (TTL vs TTT)
14. Notes on faithfulness

## 1. Setup

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from neuralrnn import AutoModel, RLTrainer, RLTrainingArguments, PPOLoss, \
    SyncVectorEnv, make_env
from neuralrnn.rl.plume_agents import build_plume_agent
from neuralrnn.rl.plume_eval import (make_eval_env, fixed_eval_grid,
                                     evaluate_agent, eval_all_datasets,
                                     sparsity_sweep, save_eval, load_eval,
                                     EVAL_DATASETS, SPARSITY_LEVELS)
from neuralrnn.rl.envs.plume import PlumeEnv
from neuralrnn.rl.vecnormalize import VecNormalize
from neuralrnn.rl.envs.plume_sim import load_or_generate_plume

MODEL_DIR = "models/s1"
FIG_DIR = "figs/s1"
PLUME_DATA_DIR = os.path.join(MODEL_DIR, "plume_data")
EVAL_DIR = os.path.join(MODEL_DIR, "eval")
for d in (MODEL_DIR, FIG_DIR, PLUME_DATA_DIR, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]  # paper: 14 seeds, top-5 reported
STAGE_STEPS = {1: 1_000_000, 2: 4_000_000}  # paper step budget per seed
DATASETS = list(EVAL_DATASETS)
REGIME_COLORS = {"TRACK": "#1b9e77", "RECOVER": "#7570b3", "SEARCH": "#d95f02"}
# reference x-axis convention for shifted [0, 1) angle units
ANGLE_TICKS = [0, 0.25, 0.5, 0.75, 1.0]
ANGLE_TICKLABELS = ["0", "-90", "$\\pm$180", "+90", "0"]
DEVICE = "cpu"  # tiny recurrent models: CPU beats GPU (kernel-launch overhead)


def savefig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"[fig] {path}")
    plt.show()


def agent_dir(seed, stage):
    if stage == "pretrain":
        return os.path.join(MODEL_DIR, f"vrnn_seed{seed}", "pretrain")
    return os.path.join(MODEL_DIR, f"vrnn_seed{seed}", f"stage{stage}", "final")


def agent_exists(seed, stage):
    return os.path.exists(os.path.join(agent_dir(seed, stage), "config.json"))


def eval_path(seed, tag):
    d = os.path.join(EVAL_DIR, f"vrnn_seed{seed}")
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{tag}.pkl")


def plot_plume_ax(ax, plume, k, centerline=False, wind_arrow=True,
                  puff_color="0.85", xlim=(-1, 10), ylim=(-3, 3)):
    """Puff scatter (+ optional centerline and wind arrow) at snapshot k."""
    x, y, r, pid = plume.puffs_at(min(int(k), plume.n_steps - 1))
    ax.scatter(x, y, s=np.clip((r * 30) ** 2, 1, 400), c=puff_color,
               edgecolors="none", zorder=1)
    if centerline and plume.has_centerline:
        cx, cy, ca = plume.centerline_at(min(int(k), plume.n_steps - 1))
        ax.plot(cx, cy, "-", c="steelblue", lw=1.2, alpha=0.8, zorder=2,
                label="centerline")
    if wind_arrow:
        wx, wy = plume.wind_at(min(int(k), plume.n_steps - 1))
        ax.annotate("", xy=(xlim[0] + 1.2 + wx, ylim[1] - 0.5 + wy),
                    xytext=(xlim[0] + 1.2, ylim[1] - 0.5),
                    arrowprops=dict(arrowstyle="->", color="k", lw=1.5))
    ax.plot(0, 0, "g^", ms=10, zorder=3)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")

### Plume analysis utilities

The helper functions below (regime segmentation, course direction relative to
wind and centerline, odor-memory features, common PCA subspace, eigenspectra
and timescales, transition times) are pure-numpy ports of the paper's
`code/log_analysis.py`. They are specific to the plume-tracking paradigm, so
they live in this notebook rather than in the package; the only package
dependency is the generic `fit_pca`.

In [ ]:
"""Plume-tracking behavior + neural analysis (Singh et al. 2023).

Pure-numpy ports of the reference ``code/log_analysis.py`` helpers
(get_traj_df / get_regimes / get_episode_metadata, without pandas) plus
neural-analysis utilities (common PCA subspace, odor-triggered states,
behavior regression, eigenspectra) that operate on the episode dicts
produced by ``neuralrnn.rl.plume_eval.evaluate_agent``.

Regime labels (per timestep, from ``odor_lastenc`` = steps since last odor
encounter): TRACK (<= RECOVER_MIN=12), SEARCH (>= RECOVER_MAX=25), RECOVER
in between; the first 12 steps of an episode are always RECOVER (warm-up).
Seed-specific threshold overrides in the reference (config.seedmeta) are not
ported; pass ``recover_min``/``recover_max`` explicitly instead.
"""
from __future__ import annotations

import numpy as np

from neuralrnn.analysis import fit_pca

ODOR_THRESHOLD = 1e-4
RECOVER_MIN = 12
RECOVER_MAX = 25


# --------------------------------------------------------------------- ports
def wind_xy_to_theta(x, y):
    """CCW angle in units of pi: +1 = +180 deg, -1 = -180 deg."""
    return np.angle(np.asarray(x) + 1j * np.asarray(y)) / np.pi


def shift_scale_theta(theta):
    """Map [-1, +1] (units of pi) to [0, 1] (0 = -180 deg)."""
    return (theta + 1.0) / 2.0


def rle(inarray):
    """Run-length encoding; returns (run_lengths, start_positions, values)."""
    ia = np.asarray(inarray)
    n = len(ia)
    if n == 0:
        return None, None, None
    y = np.array(ia[1:] != ia[:-1])
    i = np.append(np.where(y), n - 1)
    z = np.diff(np.append(-1, i))
    p = np.cumsum(np.append(0, z))[:-1]
    return z, p, ia[i]


def _count_lastenc(odor_01):
    """Steps since last odor encounter per timestep (reference _count_lenc:
    lastenc[t] counts consecutive off-plume steps up to and incl. t-1)."""
    lenc = np.zeros(len(odor_01), dtype=np.int64)
    count = 0
    for i in range(len(odor_01) - 1):
        count = 0 if odor_01[i] == 1 else count + 1
        lenc[i + 1] = count
    return lenc


def label_regimes(odor_lastenc, recover_min: int = RECOVER_MIN,
                  recover_max: int = RECOVER_MAX) -> np.ndarray:
    """TRACK / RECOVER / SEARCH labels per timestep (reference get_regimes)."""
    odor_lastenc = np.asarray(odor_lastenc)
    regime = np.full(len(odor_lastenc), "RECOVER", dtype=object)
    regime[odor_lastenc <= recover_min] = "TRACK"
    regime[odor_lastenc >= recover_max] = "SEARCH"
    between = (odor_lastenc >= recover_min) & (odor_lastenc < recover_max)
    regime[between] = "RECOVER"
    regime[:recover_min] = "RECOVER"  # warm-up
    return regime


# ------------------------------------------------------------------- frames
def episode_to_frame(ep: dict, odor_threshold: float = ODOR_THRESHOLD,
                     recover_min: int = RECOVER_MIN,
                     recover_max: int = RECOVER_MAX) -> dict:
    """Per-timestep behavioral frame of one eval episode (dict of (T,)
    arrays), the numpy counterpart of the reference ``get_traj_df``.

    Note: ``ep['loc_x']``/``loc_y`` have T+1 entries (initial position
    included); the frame uses the post-step positions, aligned with
    obs/actions/rewards.
    """
    obs = np.asarray(ep["obs"])
    actions = np.asarray(ep["actions"])  # squashed [0, 1], [step, turn]
    odor = np.asarray(obs[:, -1], dtype=np.float64)
    odor_obs = np.where(odor <= odor_threshold, 0.0, odor)
    odor_01 = (odor_obs > odor_threshold).astype(np.int64)
    lastenc = _count_lastenc(odor_01)
    wind_theta = shift_scale_theta(wind_xy_to_theta(obs[:, 0], obs[:, 1]))
    agent_theta = shift_scale_theta(
        wind_xy_to_theta(np.cos(ep["agent_angle"]), np.sin(ep["agent_angle"])))
    wind_theta_ground = shift_scale_theta(
        wind_xy_to_theta(ep["wind_x"], ep["wind_y"]))
    return {
        "loc_x": np.asarray(ep["loc_x"])[1:],
        "loc_y": np.asarray(ep["loc_y"])[1:],
        "t_val": np.asarray(ep["t_idx"], dtype=np.float64) / 25.0,
        "step": actions[:, 0], "turn": actions[:, 1],
        "wind_x_obs": obs[:, 0], "wind_y_obs": obs[:, 1],
        "wind_theta_obs": wind_theta,
        "odor_obs": odor_obs, "odor_01": odor_01,
        "odor_clip": np.clip(odor_obs, 0.0, 1.0),
        "odor_lastenc": lastenc,
        "regime": label_regimes(lastenc, recover_min, recover_max),
        "stray_distance": np.asarray(ep["stray_distance"]),
        "agent_angle_ground": agent_theta,
        "wind_angle_ground": wind_theta_ground,
        "wind_speed_ground": np.hypot(ep["wind_x"], ep["wind_y"]),
    }


def episode_metadata(ep: dict, odor_threshold: float = ODOR_THRESHOLD,
                     recover_min: int = RECOVER_MIN,
                     recover_max: int = RECOVER_MAX) -> dict:
    """Per-episode summary stats (reference ``get_episode_metadata``)."""
    fr = episode_to_frame(ep, odor_threshold, recover_min, recover_max)
    T = len(fr["loc_x"])
    off_plume = (fr["odor_obs"] <= odor_threshold).astype(np.int64)
    lengths, _, values = rle(off_plume)
    exit_durations = lengths[values == 1] if lengths is not None else []
    regimes, counts = np.unique(fr["regime"], return_counts=True)
    r_counts = dict(zip(regimes.tolist(), counts.tolist()))
    return {
        "done": ep["outcome"],
        "ep_length": T,
        "n_track": r_counts.get("TRACK", 0),
        "n_recover": r_counts.get("RECOVER", 0),
        "n_search": r_counts.get("SEARCH", 0),
        "off_plume_fraction": float(off_plume.sum() / T) if T else 0.0,
        "turn_score": float(np.mean(np.abs(fr["turn"] - 0.5))),
        "speed_score": float(np.mean(fr["step"])),
        "num_exits": int((values == 1).sum()) if values is not None else 0,
        "max_exit_duration": int(max(exit_durations))
        if len(exit_durations) else 0,
        "max_stray_distance": float(np.max(fr["stray_distance"])),
        "max_odor_lastenc": float(np.max(fr["odor_lastenc"])),
        "avg_stray_distance": float(np.mean(fr["stray_distance"])),
        "avg_odor_lastenc": float(np.mean(fr["odor_lastenc"])),
        "median_stray_distance": float(np.median(fr["stray_distance"])),
        "median_odor_lastenc": float(np.median(fr["odor_lastenc"])),
        "agent_angle_ground_median": float(np.median(fr["agent_angle_ground"])),
        "radius_final": float(np.hypot(ep["loc_x"][-1], ep["loc_y"][-1])),
        "radius_covered": float(np.hypot(ep["loc_x"][0], ep["loc_y"][0])
                                - np.hypot(ep["loc_x"][-1], ep["loc_y"][-1])),
        "end_x": float(ep["loc_x"][-1]), "end_y": float(ep["loc_y"][-1]),
        "start_x": float(ep["loc_x"][0]), "start_y": float(ep["loc_y"][0]),
        "reward": float(ep["reward_total"]),
    }


def outcome_summary(episodes: list[dict]) -> dict:
    """HOME/OOB/OOT counts + success fraction + mean reward over episodes."""
    outcomes = [ep["outcome"] for ep in episodes]
    n = len(outcomes)
    counts = {k: outcomes.count(k) for k in ("HOME", "OOB", "OOT")}
    return {"n_episodes": n, **counts,
            "success_fraction": counts["HOME"] / max(n, 1),
            "mean_reward": float(np.mean([ep["reward_total"]
                                          for ep in episodes])) if n else 0.0}


def time_to_contact(ep: dict, odor_threshold: float = ODOR_THRESHOLD) -> float:
    """Steps until the first odor encounter; ``ep['length']`` if never."""
    odor = np.asarray(ep["obs"][:, -1])
    hits = np.flatnonzero(odor > odor_threshold)
    return float(hits[0]) if len(hits) else float(ep["length"])


# ------------------------------------------------------------------- neural
def fit_common_pca(episodes: list[dict], n_components: int = 12):
    """Fit one PCA on the stacked hidden states of all episodes (reference
    ``get_pca_common``); returns a PCAResult from analysis.dimensionality."""
    H = np.vstack([np.asarray(ep["states"]) for ep in episodes])
    return fit_pca(H, n_components=n_components)


def transform_states(pca, states: np.ndarray) -> np.ndarray:
    """Project (T, M) states onto the common PCs; returns (T, n_components)."""
    return (np.asarray(states, dtype=np.float64) - pca.mean) @ pca.components.T


def odor_triggered_states(episodes: list[dict], pre: int = 10, post: int = 25,
                          odor_threshold: float = ODOR_THRESHOLD):
    """Hidden states aligned to odor onsets (odor_01 0 -> 1 transitions).

    Returns (n_events, pre + post + 1, M) array; events too close to episode
    boundaries are skipped.
    """
    snippets = []
    for ep in episodes:
        odor_01 = (np.asarray(ep["obs"][:, -1]) > odor_threshold).astype(int)
        onsets = np.flatnonzero(np.diff(odor_01) == 1) + 1
        states = np.asarray(ep["states"])
        for t0 in onsets:
            if t0 - pre >= 0 and t0 + post + 1 <= len(states):
                snippets.append(states[t0 - pre:t0 + post + 1])
    return np.asarray(snippets) if snippets else \
        np.zeros((0, pre + post + 1, 0))


def behavior_regression_r2(states: np.ndarray, targets: np.ndarray) -> np.ndarray:
    """Linear regression from (N, M) states to each column of (N, C) targets
    (with intercept); returns (C,) R^2 values. Pure numpy lstsq."""
    X = np.asarray(states, dtype=np.float64)
    Y = np.asarray(targets, dtype=np.float64)
    if Y.ndim == 1:
        Y = Y[:, None]
    X = np.concatenate([X, np.ones((X.shape[0], 1))], axis=1)
    beta, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
    pred = X @ beta
    ss_res = ((Y - pred) ** 2).sum(0)
    ss_tot = ((Y - Y.mean(0)) ** 2).sum(0)
    return 1.0 - ss_res / np.maximum(ss_tot, 1e-12)


def eigenspectrum(agent) -> np.ndarray:
    """Eigenvalues of the core recurrent weight matrix.

    CTRNN-family cores only (``h2h.weight``); the reference's architecture
    analysis (report_arch) is VRNN-only, so gated cores raise
    NotImplementedError.
    """
    core = agent.core if hasattr(agent, "core") else agent
    if hasattr(core, "h2h"):
        W = core.h2h.weight.detach().cpu().numpy()
    else:
        raise NotImplementedError(
            "eigenspectrum is defined for CTRNN-family cores (h2h.weight); "
            "the reference's architecture analysis is VRNN-only")
    return np.linalg.eigvals(W)


def eigen_timescales(eigvals: np.ndarray, dt: float = 1.0) -> np.ndarray:
    """Timescales tau = dt / |ln |lambda|| (in units of dt steps); |lambda| = 1
    gives inf (marginally stable), |lambda| -> 0 gives 0."""
    r = np.abs(np.asarray(eigvals))
    with np.errstate(divide="ignore", invalid="ignore"):
        tau = dt / np.abs(np.log(np.maximum(r, 1e-300)))
    tau[r < 1e-12] = 0.0
    return tau


# ============================================================ EWMA features
def ewma(x: np.ndarray, span: float) -> np.ndarray:
    """Exponentially weighted moving average, matching
    ``pandas.Series.ewm(span=span).mean()`` (adjust=True):
    alpha = 2 / (span + 1), y_t = sum_i (1-a)^(t-i) x_i / sum_i (1-a)^(t-i).
    """
    x = np.asarray(x, dtype=np.float64)
    alpha = 2.0 / (span + 1.0)
    y = np.empty_like(x)
    num, den = 0.0, 0.0
    for t in range(len(x)):
        num = x[t] + (1.0 - alpha) * num
        den = 1.0 + (1.0 - alpha) * den
        y[t] = num / den
    return y


def odor_onsets(odor_01: np.ndarray) -> np.ndarray:
    """Odor onset indicator: 1 at 0->1 transitions of the binarized odor
    signal (reference ``odor_01.diff().fillna(0).clip(lower=0)``)."""
    d = np.diff(np.asarray(odor_01, dtype=np.float64), prepend=0.0)
    return np.clip(d, 0.0, 1.0)


def extended_odor_features(odor_obs: np.ndarray, windows,
                           odor_threshold: float = ODOR_THRESHOLD) -> dict:
    """Windowed odor features for one episode (reference get_traj_df
    extended_metadata): for each window j, ``odor_ewm_j`` (EWMA of clipped
    odor) and ``odor_enc_j`` (EWMA of odor onsets, x25 -> encounters/s)."""
    odor_obs = np.asarray(odor_obs, dtype=np.float64)
    odor_clip = np.clip(np.where(odor_obs <= odor_threshold, 0.0, odor_obs),
                        0.0, 1.0)
    onsets = odor_onsets(odor_obs > odor_threshold)
    feats = {}
    for j in windows:
        j = int(j)
        feats[f"odor_ewm_{j}"] = ewma(odor_clip, j)
        feats[f"odor_enc_{j}"] = ewma(onsets, j) * 25.0
    return feats


def window_scan_r2(episodes: list[dict], windows=range(2, 76, 2),
                   odor_threshold: float = ODOR_THRESHOLD) -> dict:
    """R^2 of linear regression from hidden states onto windowed odor
    features, scanned over window lengths (reference report_correlations).

    Returns dict with ``windows`` (W,), ``r2_ewm`` (W,), ``r2_enc`` (W,) and
    the peak windows ``best_ewm`` / ``best_enc``.
    """
    windows = np.asarray(list(windows))
    H, feats = [], {f"odor_ewm_{int(j)}": [] for j in windows}
    feats.update({f"odor_enc_{int(j)}": [] for j in windows})
    for ep in episodes:
        H.append(np.asarray(ep["states"], dtype=np.float64))
        f = extended_odor_features(np.asarray(ep["obs"])[:, -1], windows,
                                   odor_threshold)
        for k, v in f.items():
            feats[k].append(v)
    H = np.vstack(H)
    r2_ewm, r2_enc = [], []
    for j in windows:
        j = int(j)
        r2_ewm.append(behavior_regression_r2(
            H, np.concatenate(feats[f"odor_ewm_{j}"]))[0])
        r2_enc.append(behavior_regression_r2(
            H, np.concatenate(feats[f"odor_enc_{j}"]))[0])
    r2_ewm, r2_enc = np.asarray(r2_ewm), np.asarray(r2_enc)
    return {"windows": windows, "r2_ewm": r2_ewm, "r2_enc": r2_enc,
            "best_ewm": int(windows[int(np.argmax(r2_ewm))]),
            "best_enc": int(windows[int(np.argmax(r2_enc))])}


# ============================================== course direction / centerline
def wrap_angle_pi(a):
    """Wrap angle(s) to [-pi, pi)."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def course_direction_frame(ep: dict, plume=None,
                           centerline_half_width: float = 0.02,
                           trim_wind_deg: float | None = None) -> dict:
    """Course/head direction (CD/HD) of one episode relative to wind and
    centerline.

    Port of the reference ``report_regime_dists`` CD/HD pipeline. The course
    direction is the direction of the agent's instantaneous displacement
    (ground frame). Relative angles follow the reference convention:

        u_rel = mod((theta_agent - theta_ref) / (2*pi), 1)   in [0, 1)

    (``roll_around_angle(agent_angle_ground - (ref_angle - 0.5))`` with the
    reference's shift-scaled angles), where ``theta_ref`` is the current wind
    or local centerline angle. Plot with x-ticks [0, .25, .5, .75, 1] labeled
    [0, -90, +/-180, +90, 0]: 0.5 (+/-180 deg, anti-parallel) is where TRACK
    peaks, since the wind / centerline line orientation points downwind while
    the agent flies upwind. Note the centerline angle is a LINE orientation
    (arctan(dy/dx), mod 180 deg), not a directed angle -- see
    plume_sim.generate_centerline.

    The local centerline angle is the median tangent of centerline points
    within ``centerline_half_width`` in x of the agent (0.02 m in the
    reference). With ``trim_wind_deg`` (15 deg for switch-once in the
    reference), timesteps whose absolute wind angle is below the threshold
    are masked out (pre-switch timesteps).

    Returns dict of (T,) arrays in both radians and reference [0,1) units
    (``*_u`` suffix); ``mask`` marks valid timesteps.
    """
    loc_x = np.asarray(ep["loc_x"])[1:]
    dx = np.diff(np.asarray(ep["loc_x"]))
    dy = np.diff(np.asarray(ep["loc_y"]))
    T = len(loc_x)
    dx, dy = dx[:T], dy[:T]
    course = np.arctan2(dy, dx)
    head = np.asarray(ep["agent_angle"], dtype=np.float64)[:T]
    wind = np.arctan2(np.asarray(ep["wind_y"])[:T], np.asarray(ep["wind_x"])[:T])

    valid = np.hypot(dx, dy) > 1e-9  # undefined CD when not moving
    if trim_wind_deg is not None:
        valid &= np.abs(np.degrees(wind)) >= trim_wind_deg

    if plume is not None and plume.has_centerline:
        tidx = np.asarray(ep["t_idx"])[:T]
        # map global tidx to local snapshot index
        kmap = {int(t): i for i, t in enumerate(plume.tidxs)}
        cl_angle = np.array([
            plume.centerline_angle_at(kmap[int(t)], x, centerline_half_width)
            if int(t) in kmap else np.nan
            for t, x in zip(tidx, loc_x)])
        valid &= ~np.isnan(cl_angle)
    else:
        cl_angle = np.full(T, np.nan)

    def rel_u(theta, ref):
        return np.mod((theta - ref) / (2 * np.pi), 1.0)

    return {"course_angle": course, "head_angle": head, "wind_angle": wind,
            "centerline_angle": cl_angle,
            "cd_rel_wind": np.mod(course - wind, 2 * np.pi),
            "cd_rel_centerline": np.mod(course - cl_angle, 2 * np.pi),
            "cd_rel_wind_u": rel_u(course, wind),
            "cd_rel_centerline_u": rel_u(course, cl_angle),
            "hd_rel_wind_u": rel_u(head, wind),
            "hd_rel_centerline_u": rel_u(head, cl_angle),
            "mask": valid}


# ------------------------------------------------------------- regime stats
def regime_behavior_samples(episodes: list[dict], balance_outcomes: bool = True,
                            n_per_outcome: int | None = None,
                            odor_threshold: float = ODOR_THRESHOLD,
                            recover_min: int = RECOVER_MIN,
                            recover_max: int = RECOVER_MAX,
                            seed: int = 0) -> dict:
    """Per-regime behavioral samples aggregated over episodes (reference
    report_regime_dists / regime_histos): head direction (deg, 0 = downwind),
    per-step displacement dx, dy (m), turn and move actions, stray distance.

    With ``balance_outcomes`` an equal number of HOME and non-HOME episodes
    is used (the reference balances successful and unsuccessful episodes).
    """
    rng = np.random.default_rng(seed)
    eps = list(episodes)
    if balance_outcomes:
        homed = [e for e in eps if e["outcome"] == "HOME"]
        other = [e for e in eps if e["outcome"] != "HOME"]
        n = n_per_outcome or min(len(homed), len(other))
        if len(homed) > n:
            idx = rng.choice(len(homed), size=n, replace=False)
            homed = [homed[i] for i in idx]
        if len(other) > n:
            idx = rng.choice(len(other), size=n, replace=False)
            other = [other[i] for i in idx]
        eps = homed + other

    out = {r: {k: [] for k in ("head_direction", "dx", "dy", "turn",
                               "move", "stray_distance")}
           for r in ("TRACK", "RECOVER", "SEARCH")}
    for ep in eps:
        fr = episode_to_frame(ep, odor_threshold, recover_min, recover_max)
        loc_x = np.asarray(ep["loc_x"])
        loc_y = np.asarray(ep["loc_y"])
        dx = np.diff(loc_x)[:len(fr["loc_x"])]
        dy = np.diff(loc_y)[:len(fr["loc_y"])]
        hd = np.degrees(np.asarray(ep["agent_angle"])[:len(dx)])
        for r in out:
            m = fr["regime"] == r
            if not m.any():
                continue
            out[r]["head_direction"].append(hd[m])
            out[r]["dx"].append(dx[m])
            out[r]["dy"].append(dy[m])
            out[r]["turn"].append(np.asarray(fr["turn"])[m])
            out[r]["move"].append(np.asarray(fr["step"])[m])
            out[r]["stray_distance"].append(np.asarray(fr["stray_distance"])[m])
    for r in out:
        for k in out[r]:
            out[r][k] = (np.concatenate(out[r][k]) if out[r][k]
                         else np.zeros(0))
    return out


# ------------------------------------------------------------- eigen / Jacobians
def sorted_taus(eigvals: np.ndarray, fix_unstable: bool = True) -> np.ndarray:
    """Sorted (desc) integration timescales tau = 1/|ln|lambda|| in steps;
    with ``fix_unstable`` the tau of |lambda| >= 1 modes is set to 1
    (reference arch_utils.get_taus, filters=['fix'])."""
    r = np.abs(np.asarray(eigvals))
    with np.errstate(divide="ignore", invalid="ignore"):
        tau = np.abs(1.0 / np.log(np.maximum(r, 1e-300)))
    if fix_unstable:
        tau[r >= 1.0] = 1.0
    return np.sort(tau)[::-1]


def recurrence_jacobians(agent, states: np.ndarray) -> np.ndarray:
    """Per-timestep recurrence Jacobian J_rec(t) = d h_t / d h_{t-1} along a
    recorded hidden-state trajectory (T, M) of a tanh CTRNN-family core
    (alpha = 1): J = diag(1 - h_t^2) @ W_hh. Returns (T, M, M)."""
    core = agent.core if hasattr(agent, "core") else agent
    if not hasattr(core, "h2h"):
        raise NotImplementedError(
            "recurrence_jacobians is defined for CTRNN-family cores")
    W = core.h2h.weight.detach().cpu().numpy().astype(np.float64)
    h = np.asarray(states, dtype=np.float64)
    return (1.0 - h ** 2)[:, :, None] * W[None, :, :]


def episode_timescale_curves(agent, episodes: list[dict]) -> np.ndarray:
    """Sorted tau curves (T_total, M) over all timesteps of all episodes
    (reference arch_utils.get_eig_df_episode + get_taus)."""
    curves = []
    for ep in episodes:
        J = recurrence_jacobians(agent, ep["states"])
        for t in range(J.shape[0]):
            curves.append(sorted_taus(np.linalg.eigvals(J[t])))
    return np.asarray(curves)


def limit_cycle_period(states: np.ndarray, max_period: int = 50,
                       tail: int | None = None,
                       measure: str = "mean_abs"):
    """Detect a quasi-limit-cycle period in a hidden-state trajectory
    (reference report_common_subspace get_period): period = argmin over p in
    [1, max_period] of a distance between h_t and h_{t-p} over the last
    ``tail`` steps (default: whole trajectory).

    ``measure``: "mean_abs" = mean |h_t - h_{t-p}|; "signed_sum" = the
    reference's exact measure, sum of |sum_over_units(h_t - h_{t-p})| over the
    tail (coherent drift is penalized; only true oscillation dips).
    Note: the reference's `argmin + 2` indexing quirk is NOT replicated
    (correct indexing is argmin + 1 for p in [1, max_period]).

    Returns (period, dists_per_p)."""
    h = np.asarray(states, dtype=np.float64)
    if tail is not None:
        h = h[-(tail + max_period):] if measure == "signed_sum" else h[-tail:]
    dists = []
    for p in range(1, max_period + 1):
        if len(h) <= p:
            dists.append(np.inf)
            continue
        d = h[p:] - h[:-p]
        if measure == "signed_sum":
            # diff over the segment first, then score the last `tail` entries
            d = np.abs(d.sum(axis=1))
            d = d[-tail:] if tail is not None else d
            dists.append(float(d.sum()))
        else:
            dists.append(float(np.mean(np.abs(d))))
    dists = np.asarray(dists)
    return int(np.argmin(dists) + 1), dists


# --------------------------------------------------------------- centroids/TTC
def outcome_centroids(episodes: list[dict], n_tail: int = 25,
                      min_oob_len: int = 75) -> dict:
    """HOME / OOB neural centroids (reference report_ttcs): mean of the last
    ``n_tail`` hidden states over HOME episodes, resp. OOB episodes longer
    than ``min_oob_len`` steps; ``range`` = half the inter-centroid distance.
    """
    homed = [np.asarray(e["states"])[-n_tail:] for e in episodes
             if e["outcome"] == "HOME"]
    oob = [np.asarray(e["states"])[-n_tail:] for e in episodes
           if e["outcome"] == "OOB" and e["length"] > min_oob_len]
    if not homed or not oob:
        raise ValueError("need both HOME and long OOB episodes for centroids")
    c_home = np.vstack(homed).mean(0)
    c_oob = np.vstack(oob).mean(0)
    return {"home": c_home, "oob": c_oob,
            "range": float(np.linalg.norm(c_home - c_oob) / 2.0)}


def transition_times(episodes: list[dict], centroids: dict,
                     odor_threshold: float = ODOR_THRESHOLD,
                     min_lost_steps: int = 13) -> dict:
    """Neural-regime transition asymmetry (reference report_ttcs).

    TTL (time-to-lost): from the FINAL plume exit to the first step within
    ``centroids['range']`` of the OOB centroid (episodes with >= 2 on-plume
    steps before the exit and >= 25 steps after it).
    TTT (time-to-track): from an odor re-encounter after >=
    ``min_lost_steps`` off-plume steps to the first step within range of the
    HOME centroid.

    Returns dict with ``ttl`` and ``ttt`` lists (in steps).
    """
    ttl, ttt = [], []
    for ep in episodes:
        states = np.asarray(ep["states"])
        odor_01 = (np.asarray(ep["obs"])[:, -1] > odor_threshold).astype(int)
        T = len(odor_01)
        # TTL: final plume exit
        on_idx = np.flatnonzero(odor_01)
        if ep["outcome"] == "OOB" and len(on_idx) >= 2:
            exit_t = on_idx[-1] + 1
            if exit_t < T and (T - exit_t) >= 25:
                d = np.linalg.norm(states - centroids["oob"], axis=1)
                hits = np.flatnonzero(d[exit_t:] < centroids["range"])
                if len(hits):
                    ttl.append(int(hits[0]))
        # TTT: re-encounter after a long absence
        lastenc = _count_lastenc(odor_01)
        re = np.flatnonzero((odor_01 == 1) &
                            (np.concatenate([[0], lastenc[:-1]])
                             >= min_lost_steps))
        d = np.linalg.norm(states - centroids["home"], axis=1)
        for t0 in re:
            hits = np.flatnonzero(d[t0:] < centroids["range"])
            if len(hits):
                ttt.append(int(hits[0]))
    return {"ttl": ttl, "ttt": ttt}


# ---------------------------------------------------- concentration profiles
def concentration_at_points(px, py, pr, xs, ys) -> np.ndarray:
    """Odor concentration at query points (xs, ys): sum of (0.01/r)^3 over
    puffs whose axis-aligned square contains the point (reference
    get_concentration_at_tidx). Vectorized over query points."""
    px = np.asarray(px, dtype=np.float64)
    py = np.asarray(py, dtype=np.float64)
    pr = np.asarray(pr, dtype=np.float64)
    c = (0.01 / np.maximum(pr, 1e-9)) ** 3
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    hit = (np.abs(px[None, :] - xs[:, None]) < pr[None, :]) & \
          (np.abs(py[None, :] - ys[:, None]) < pr[None, :])
    return hit @ c

## 2. The plume simulation and the environment

**The simulated plume.** Odor is a set of discrete puffs released from a
source at the origin with Poisson rate 1.0 per step (100 Hz simulation).
Each puff grows at 0.01 m/s and is advected by the wind with additional
Gaussian cross-wind diffusion, which produces the two signature properties of
a turbulent plume: *intermittency* (an agent inside the time-averaged plume
still experiences long blanks) and *meandering* (the plume bends when the
wind shifts). The environment replays a precomputed simulation at 25 Hz
(`env_dt = 0.04 s`, 300 steps = 12 s per episode); `load_or_generate_plume`
caches simulations under `models/s1/plume_data/`, including a deterministic
plume **centerline** used later for the course-direction analysis.

Three wind regimes (the paper's non-stationary conditions) are used:

| dataset | wind | plume shape |
|---|---|---|
| `constantx5b5` | constant 0.5 m/s along +x | straight |
| `switch45x5b5` | rotates +45° once at ~58 s | bent at the switch |
| `noisy3x5b5` | continuously fluctuating | meandering |

**The POMDP.** The agent never sees the plume, its own position, or the
source. It observes three egocentric channels,

$$o_t = [\,\tilde w_x,\ \tilde w_y,\ c\,]_t,$$

where $(\tilde w_x, \tilde w_y)$ is the wind relative to the agent's own
motion, rotated into its heading frame, and $c$ is the odor concentration at
the agent, the sum of $(0.01/r)^3$ over puffs whose box contains it, clipped
to $[0, 1]$. It acts with two continuous controls $a_t = [\text{move},
\text{turn}]$, squashed to $[0, 1]$ inside the environment (max forward
speed 2.5 m/s, max turn rate $6.25\pi$ rad/s in the paper's units).

**Reward and termination.** Reaching within 0.2 m of the source ends the
episode as HOME with a +101 bonus. Every step costs $-10/300$ (five times
more while off the plume), and a radial shaping term rewards getting closer
to the source, so faster, straighter approaches score higher. Episodes also
end on OOB (straying more than 2 m from the plume, with a penalty) or on
timeout (OOT). This reward structure is the "ecological pressure" that makes
hugging the plume worthwhile: an agent that ignores the odor and rushes
upwind pays the off-plume cost.

The first plot below shows puff snapshots of the three datasets; the second
rolls out an **untrained** agent for one episode so we can see the raw
observation stream the network starts from.

In [ ]:
# Puff snapshots for the three datasets (rows) at three times (columns)
TIMES = {"constantx5b5": [62.0, 66.0, 70.0],
         "switch45x5b5": [55.0, 60.0, 65.0],   # wind switch at ~58 s
         "noisy3x5b5": [62.0, 66.0, 70.0]}
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for row, name in enumerate(DATASETS):
    plume = load_or_generate_plume(name, PLUME_DATA_DIR)
    for col, t in enumerate(TIMES[name]):
        ax = axes[row, col]
        k = int(np.argmin(np.abs(plume.times - t)))
        plot_plume_ax(ax, plume, k, centerline=True)
        wx, wy = plume.wind_at(k)
        ax.set_title(f"{name}  t={plume.times[k]:.1f}s  "
                     f"wind=({wx:.2f},{wy:.2f}) m/s", fontsize=9)
        if col == 0:
            ax.set_ylabel("y (m)")
axes[0, 1].legend(fontsize=8, loc="lower right")
fig.suptitle("Plume datasets: puff snapshots + centerline + wind arrow")
fig.tight_layout()
savefig(fig, "plume_snapshots.png")

In [ ]:
# Roll out an UNTRAINED agent to see the env interface
plume = load_or_generate_plume("constantx5b5", PLUME_DATA_DIR)
env = PlumeEnv(dataset="constantx5b5", data_dir=PLUME_DATA_DIR,
               squash_action=True, flipping=False, odor_scaling=False, seed=3)
demo_agent = build_plume_agent("vrnn", seed=3)
demo_agent.eval()

obs, _ = env.reset()
z = demo_agent.init_state(1, DEVICE)
done = torch.ones(1)
log = {k: [] for k in ("obs", "reward", "loc")}
with torch.no_grad():
    for t in range(300):
        x_t = torch.as_tensor(obs, dtype=torch.float32).view(1, -1)
        a, lp, ent, v, z = demo_agent.step(x_t, z, done)
        obs, r, term, trunc, info = env.step(a.numpy()[0])
        done = torch.tensor([float(term or trunc)])
        log["obs"].append(obs.copy()); log["reward"].append(r)
        log["loc"].append(info["location"].copy())
        if term or trunc:
            break
obs_arr, locs = np.array(log["obs"]), np.array(log["loc"])
print(f"outcome={info['done']} steps={len(locs)} return={sum(log['reward']):.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
k_last = int(np.argmin(np.abs(plume.tidxs - info["t_idx"])))
plot_plume_ax(ax, plume, k_last, wind_arrow=False)
ax.plot(locs[:, 0], locs[:, 1], "r.-", ms=2, lw=1, label="agent", zorder=4)
ax.plot(*locs[0], "k*", ms=12, label="start", zorder=5)
ax.set_title(f"untrained VRNN rollout ({info['done']})")
ax.legend()
ax = axes[1]
tt = np.arange(len(obs_arr)) * env.dt
ax.plot(tt, obs_arr[:, 0], label="wind_x (ego)")
ax.plot(tt, obs_arr[:, 1], label="wind_y (ego)")
ax.plot(tt, obs_arr[:, 2], label="odor", lw=2)
ax.set_xlabel("t (s)"); ax.set_title("egocentric observations"); ax.legend()
fig.tight_layout()
savefig(fig, "env_demo_rollout.png")

## 3. The agent: a vanilla tanh RNN actor-critic

`build_plume_agent("vrnn", seed)` assembles the paper's main model from
framework pieces. The recurrent core is `CTRNNConfig(activation="tanh",
dt=None)`, i.e. Euler step size equal to the time constant ($\alpha = 1$), so
the update is exactly a vanilla RNN,

$$h_t = \tanh(W_{hh}\, h_{t-1} + W_x\, o_t + b), \qquad h_t \in \mathbb{R}^{64}.$$

On top of the shared core sit two separate 2-layer tanh MLPs (64, 64): the
**actor** outputs the mean of a diagonal Gaussian policy with a learned,
state-independent log-standard-deviation, and the **critic** outputs the
scalar value estimate $V(o_{\le t})$. The critic is only used during
training; behavior comes from the actor.

Two design choices of the paper matter for everything after:

* **Vanilla RNN, not GRU/LSTM.** The authors deliberately avoid gating so
  that the trained network stays analyzable with dynamical-systems tools:
  the eigenspectrum of $W_{hh}$ (section 12) and the recurrence Jacobians
  along trajectories are only simple to interpret for a vanilla core. Their
  appendix shows a GRU does not perform better anyway.
* **Initialization.** The core is drawn from $N(0, 1/\sqrt{64})$ with zero
  biases (applied by `init_plume_agent_`), and the heads use the reference's
  orthogonal scheme (gain $\sqrt{2}$ for hidden layers, 0.01 for the actor
  output, 1.0 for the critic output).

We also save the *pre-training* weights of every seed, so section 12 can
compare the eigenspectrum before and after training.

In [ ]:
for seed in SEEDS:
    if not agent_exists(seed, "pretrain"):
        agent = build_plume_agent("vrnn", seed=seed)
        agent.save_pretrained(agent_dir(seed, "pretrain"))
        print(f"[pretrain] saved seed{seed}")

_a = build_plume_agent("vrnn", seed=0)
print(f"vrnn: input_dim={_a.config.input_dim} "
      f"latent_dim={_a.config.latent_dim} "
      f"params={sum(p.numel() for p in _a.parameters())}")

## 4. Two-stage PPO training

The agent is trained with proximal policy optimization. The policy is updated
by gradient ascent on the clipped surrogate plus value and entropy terms,

$$\mathcal{L}(\theta) = \mathbb{E}_t\Big[ \min\big(\rho_t(\theta) A_t,\
\mathrm{clip}(\rho_t(\theta), 1\!-\!\varepsilon, 1\!+\!\varepsilon) A_t\big)
- c_1 \big(V_\theta(s_t) - R_t\big)^2 + c_2\, \mathcal{S}[\pi_\theta](s_t) \Big],$$

with $\rho_t(\theta) = \pi_\theta(a_t|s_t)/\pi_{\theta_{old}}(a_t|s_t)$,
$\varepsilon = 0.2$, $c_1 = 0.5$, $c_2 = 0.005$, and advantages $A_t$ from
generalized advantage estimation ($\gamma = 0.99$, $\lambda = 0.95$). The
paper's optimization details are kept exactly: learning rate $3\times10^{-4}$
with linear annealing, 10 epochs of 4 minibatches per update (one environment
trajectory per minibatch, the recurrent generator of the reference), clipped
value loss, gradient-norm clip 0.5, weight decay $10^{-4}$, advantages
normalized over the whole rollout, and batches of 4 envs x 2048 steps = 8192
steps per update. As in the reference implementation, training runs behind
`VecNormalize`: observations are normalized by running statistics and rewards
are scaled by the discounted-return standard deviation, both clipped at
$\pm 10$, with fresh statistics per stage.

**Curriculum.** Training directly on the meandering plume is unstable, so the
paper uses two stages:

| stage | plume | odor density | starting region | env steps |
|---|---|---|---|---|
| 1 | `constantx5b5` (constant wind) | sparse (`birthx = 0.3`) | wide (`qvar = 2.0`, difficulty 0.7-0.8) | 1M |
| 2 | `noisy3x5b5` (fluctuating wind) | denser (`birthx = 0.8`) | tighter (`qvar = 0.5`, difficulty 0.4-0.8) | 4M |

Both stages train with three augmentations from the reference: `squash_action`
(the env applies the squashing), `flipping` (episodes are mirrored with
probability 1/2, doubling the effective plume orientations), and
`odor_scaling` (odor gain resampled from $U(0.5, 1.5)$ per episode).

**Load-first.** Existing `stage{k}/final` checkpoints under `models/s1/` are
reused, so by default the cell below trains nothing. To retrain, delete the
corresponding folder (a full 5-seed run takes about 10 h on CPU; see
`tests/plume/train_plume.py` and `run_all_seeds.sh`). The training curves
plot the smoothed episode return of both stages for every seed; the jump in
difficulty at the stage boundary (vertical line, 1M steps) is clearly visible
as a dip before the stage-2 recovery.

In [ ]:
STAGE_KWARGS = {
    1: dict(dataset="constantx5b5", birthx=0.3, qvar=2.0,
            diff_min=0.7, diff_max=0.8),
    2: dict(dataset="noisy3x5b5", birthx=0.8, qvar=0.5,
            diff_min=0.4, diff_max=0.8),
}


def train_one(seed, stage, num_envs=4, num_steps=2048):
    cfg = STAGE_KWARGS[stage]
    if stage == 1:
        agent = build_plume_agent("vrnn", seed=seed)
    else:
        agent = AutoModel.from_pretrained(agent_dir(seed, 1))
    agent.train()
    env_kwargs = dict(dataset=cfg["dataset"], data_dir=PLUME_DATA_DIR,
                      squash_action=True, flipping=True, odor_scaling=True,
                      birthx=cfg["birthx"], qvar=cfg["qvar"],
                      diff_min=cfg["diff_min"], diff_max=cfg["diff_max"])
    envs = SyncVectorEnv([make_env("plume", seed=1000 * seed + i, **env_kwargs)
                          for i in range(num_envs)])
    envs = VecNormalize(envs, gamma=0.99)  # reference: SB3 VecNormalize
    args = RLTrainingArguments(
        total_timesteps=STAGE_STEPS[stage], num_envs=num_envs,
        num_steps=num_steps, num_minibatches=4, update_epochs=10,
        gamma=0.99, estimator="gae", gae_lambda=0.95, lr=3e-4,
        anneal_lr=True, weight_decay=1e-4, max_grad_norm=0.5,
        adv_norm="rollout",  # reference: rollout-global adv normalization
        seed=seed, device=DEVICE,
        output_dir=os.path.join(MODEL_DIR, f"vrnn_seed{seed}", f"stage{stage}"))
    loss = PPOLoss(clip_coef=0.2, vf_coef=0.5, ent_coef=0.005,
                   clip_vloss=True, norm_adv=False)
    history = RLTrainer(agent, envs, args, loss=loss).train()
    envs.close()
    return history


# Train what is missing (default: nothing, if checkpoints were pre-trained).
for seed in SEEDS:
    for stage in (1, 2):
        if agent_exists(seed, stage):
            continue
        if stage == 2 and not agent_exists(seed, 1):
            train_one(seed, 1)
        print(f"[train] seed{seed} stage{stage}")
        train_one(seed, stage)

In [ ]:
# Training curves (from the trainers' history.json)
fig, ax = plt.subplots(figsize=(8, 4.5))
for seed in SEEDS:
    for stage, ls in ((1, "-"), (2, "--")):
        hpath = os.path.join(MODEL_DIR, f"vrnn_seed{seed}",
                             f"stage{stage}", "history.json")
        if not os.path.exists(hpath):
            continue
        with open(hpath) as f:
            hist = json.load(f)
        steps = np.array([s["global_step"] for s in hist["scalars"]])
        rets = np.array([s["episodic_return"] for s in hist["scalars"]],
                        dtype=float)
        w = 5
        sm = np.convolve(np.nan_to_num(rets, nan=np.nanmean(rets)),
                         np.ones(w) / w, mode="same")
        ax.plot(steps + (STAGE_STEPS[1] if stage == 2 else 0), sm, ls,
                alpha=0.85, label=f"seed{seed} stage{stage}")
ax.axvline(STAGE_STEPS[1], color="k", lw=0.5, alpha=0.4)
ax.set_xlabel("env steps"); ax.set_ylabel("episode return (smoothed)")
ax.set_title("PPO two-stage curriculum (stage 1: constant sparse -> "
             "stage 2: noisy dense)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
savefig(fig, "training_curves.png")

## 5. Fixed-grid evaluation and seed selection

The paper's assay replaces random episode starts with a **fixed grid of 240
episodes** per dataset: 3 start x-positions (4, 6, 8 m) x 2 time offsets x 5
y-positions spanning the plume width x 8 initial headings. The policy is run
deterministically (Gaussian mean, no sampling), training augmentations are
off, and the plume is dense (`birthx = 1.0`). `switch45` episodes start at
t >= 58 s so that every episode straddles the wind switch. Observations are
still online-normalized during evaluation (the reference wraps the eval env
in a fresh, still-updating `VecNormalize`). Eval logs are cached under
`models/s1/eval/`, so this section also runs load-first.

A second assay tests **sparsity generalization**: the constant plume is
thinned at load time (`birthx_max` = 0.8, 0.6, 0.4, 0.2), making odor
encounters rarer than anything seen in training.

The paper trains 14 seeds and reports the top 5. With 5 seeds we report all
of them and select the **top seed** by HOME fraction on `constantx5b5` for
the single-agent analyses below. The printed table is the success fraction
(HOME episodes out of 240) per seed and dataset.

In [ ]:
for seed in SEEDS:
    if not agent_exists(seed, 2):
        print(f"[eval] SKIP seed{seed}: no stage2 checkpoint")
        continue
    need = [d for d in DATASETS if not os.path.exists(eval_path(seed, d))]
    need_sparse = not os.path.exists(eval_path(seed, "sparse"))
    if not need and not need_sparse:
        continue
    agent = AutoModel.from_pretrained(agent_dir(seed, 2))
    if need:
        results = eval_all_datasets(agent, need, data_dir=PLUME_DATA_DIR,
                                    seed=seed, progress=True)
        for d, eps in results.items():
            save_eval(eval_path(seed, d), eps)
    if need_sparse:
        sweep = sparsity_sweep(agent, "constantx5b5", data_dir=PLUME_DATA_DIR,
                               seed=seed, progress=True)
        save_eval(eval_path(seed, "sparse"), sweep)
    print(f"[eval] seed{seed} done")

In [ ]:
print("success fraction (HOME / 240 episodes):")
print("        " + " ".join(f"{d:>14s}" for d in DATASETS))
TOP, best_score = None, -1.0
for seed in SEEDS:
    row = []
    for d in DATASETS:
        p = eval_path(seed, d)
        if not os.path.exists(p):
            row.append("      --      ")
            continue
        s = outcome_summary(load_eval(p))
        row.append(f"{s['success_fraction']:>14.3f}")
        if d == "constantx5b5" and s["success_fraction"] > best_score:
            TOP, best_score = seed, s["success_fraction"]
    print(f"  seed{seed} " + " ".join(row))
print(f"top seed: {TOP}")
with open(os.path.join(EVAL_DIR, "top_seeds.json"), "w") as f:
    json.dump({"vrnn": TOP}, f, indent=2)


def load_top(tag, seed=None):
    return load_eval(eval_path(TOP if seed is None else seed, tag))

## 6. Success across wind regimes and plume sparsity

The first result: the trained agents solve the constant plume and, without
ever having trained on them, generalize to the non-stationary `switch45` and
`noisy3` regimes. One honest caveat about scale: the paper trains 14 seeds
and reports the top 5, while we train 5 and report all of them, so our means
are pulled down by the weak seeds that the paper's selection would discard.
The spread across seeds is itself a finding (PPO on this task is bistable:
some seeds learn the full strategy, some stall near zero), and the strong
seeds behave as the paper describes: our top seed reaches 0.62 HOME fraction
on the constant plume, and the best switch45 seed scores 0.61 on a wind
regime it never saw in training. Performance then degrades gracefully as the
plume is sparsified below the training density (top seed: 0.51 at 0.8x
density, still 0.20 at 0.2x): the strategy tolerates much longer odor-free
gaps than it was trained on, which is where a purely reactive agent would get
lost.

In [ ]:
NICE = {"constantx5b5": "constant", "switch45x5b5": "switch 45 deg",
        "noisy3x5b5": "noisy"}
succ = {d: [] for d in DATASETS}
for seed in SEEDS:
    for d in DATASETS:
        try:
            eps = load_eval(eval_path(seed, d))
        except FileNotFoundError:
            continue
        succ[d].append(outcome_summary(eps)["success_fraction"])

fig, ax = plt.subplots(figsize=(6.5, 4.2))
xs = np.arange(len(DATASETS))
means = [np.mean(succ[d]) if succ[d] else np.nan for d in DATASETS]
ax.bar(xs, means, color="steelblue", alpha=0.75)
for i, d in enumerate(DATASETS):
    ax.scatter(np.full(len(succ[d]), i) + 0.26, succ[d], c="k", s=20,
               zorder=3)
    ax.text(i, means[i] + 0.03, f"{means[i]:.2f}", ha="center", fontsize=10)
ax.set_xticks(xs, [NICE[d] for d in DATASETS], fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel("HOME fraction (240-episode grid)")
ax.set_title("Success across wind regimes\n"
             "(bars: mean over seeds; dots: individual seeds)", fontsize=10)
fig.tight_layout()
savefig(fig, "fig4_success.png")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for seed in SEEDS:
    try:
        sweep = load_eval(eval_path(seed, "sparse"))
    except FileNotFoundError:
        continue
    levels = sorted(sweep.keys(), reverse=True)
    vals = [outcome_summary(sweep[bx])["success_fraction"]
            for bx in levels]
    ax.plot(levels, vals, "o-", alpha=0.7, label=f"seed {seed}")
    ax.plot([1.0], [outcome_summary(
        load_eval(eval_path(seed, "constantx5b5")))["success_fraction"]],
        "s", ms=7, color=ax.lines[-1].get_color())
ax.set_xlabel("plume density (birth fraction; 1.0 = evaluation density)")
ax.set_ylabel("HOME fraction")
ax.set_title("Generalization to sparser plumes (constant wind)", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "sparsity_generalization.png")

## 7. Example trajectories

Before any quantification, look at what the behavior actually is. One
evaluation episode per dataset (a HOME episode when one exists), drawn on top
of the plume and colored by the behavioral regime of each timestep (defined
in the next section). Two things are already visible: the switch45 trajectory
follows the **bent** plume around the kink left by the wind switch, and the
noisy3 trajectory follows the meandering centerline. Whatever the agent is
tracking, it is shaped like the plume, not like the current wind vector;
section 9 makes this precise.

In [ ]:
def pick_episode(eps, prefer="HOME"):
    homed = [e for e in eps if e["outcome"] == prefer]
    if homed:
        homed.sort(key=lambda e: e["length"])
        return homed[len(homed) // 2]
    return max(eps, key=lambda e: e["length"])


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, d in zip(axes, DATASETS):
    eps = load_top(d)
    ep = pick_episode(eps)
    fr = episode_to_frame(ep)
    plume = load_or_generate_plume(d, PLUME_DATA_DIR)
    # ep["t_idx"] is the GLOBAL sim index; map to the local snapshot index
    k = int(np.argmin(np.abs(plume.tidxs - ep["t_idx"][-1])))
    plot_plume_ax(ax, plume, k, centerline=True, wind_arrow=False)
    lx, ly = np.asarray(ep["loc_x"]), np.asarray(ep["loc_y"])
    for r, c in REGIME_COLORS.items():
        m = fr["regime"] == r
        ax.scatter(lx[1:][m], ly[1:][m], s=10, c=c, label=r, zorder=4)
    ax.plot(lx, ly, "-", c="0.5", lw=0.7, zorder=3)
    ax.plot(lx[0], ly[0], "k*", ms=12, zorder=5)
    ax.set_title(f"{d} ({ep['outcome']}, {ep['length']} steps)", fontsize=10)
axes[0].set_ylabel("y (m)")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle(f"Example trajectories colored by behavioral regime "
             f"(top seed {TOP})")
fig.tight_layout()
savefig(fig, "trajectories.png")

## 8. Behavioral regimes: TRACK / RECOVER / SEARCH

The paper's first core result is that the complex continuous trajectories
decompose into three behavioral modules. The decomposition is not hand-picked
by looking at paths: it is defined by a single internal variable, the number
of steps since the last odor encounter $T_{last}$, an "odour-loss activated
internal clock" in the terminology of the insect literature:

| regime | $T_{last}$ | behavior |
|---|---|---|
| TRACK | <= 12 steps (0.5 s) | upwind surge with small stray distance; zig-zag at the plume edge ("plume skimming": repeated brief exits and re-entries that keep the plume in contact) |
| RECOVER | 12-25 steps (0.5-1 s) | cross-wind casting with growing stray, still biased upwind: an active attempt to reacquire the plume |
| SEARCH | >= 25 steps (1 s) | large lateral excursions with a turning bias and little upwind progress: a different motor program altogether |

(The first 12 steps of every episode are labeled RECOVER as a warm-up. The
paper tunes the lost threshold per agent, 25-38 steps; we keep the reference
defaults 12/25 for all seeds.) These three regimes correspond to the surge /
cast / search modules of flying insects, and they were never programmed: PPO
discovered them because the clock variable is what the policy needs.

The first plot segments one episode by regime; the second shows that the
regimes are genuinely different motor regimes by comparing per-timestep
distributions of head direction, upwind and lateral displacement, the two
action channels, and stray distance, balanced between successful and
unsuccessful episodes as in the paper.

In [ ]:
eps = load_top("constantx5b5")
ep = pick_episode(eps)
fr = episode_to_frame(ep)
tt = np.arange(len(fr["odor_obs"])) * 0.04
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
ax = axes[0]
ax.plot(tt, fr["odor_clip"], "k", lw=1.2)
ax.set_ylabel("odor (clipped)")
ax.set_title(f"Regime segmentation of one episode "
             f"({ep['outcome']}, constantx5b5)")
ax = axes[1]
ax.plot(tt, fr["odor_lastenc"], "k", lw=1.2)
ax.axhline(12, color="0.5", ls="--", lw=0.8)
ax.axhline(25, color="0.5", ls="--", lw=0.8)
for r, c in REGIME_COLORS.items():
    m = fr["regime"] == r
    ax.scatter(tt[m], fr["odor_lastenc"][m], s=6, c=c, label=r)
ax.set_ylabel("steps since last odor")
ax.legend(fontsize=8, ncol=3)
ax = axes[2]
ax.plot(tt, fr["step"], label="move")
ax.plot(tt, fr["turn"], label="turn")
ax.set_xlabel("t (s)"); ax.set_ylabel("action")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "regime_segmentation.png")

In [ ]:
# Per-regime behavioral distributions (HOME/non-HOME balanced, as in the paper)
samples = regime_behavior_samples(eps, balance_outcomes=True)
PANELS = [("head_direction", "head direction", "deg; +/-180 = upwind",
           (-180, 180), 36),
          ("dx", "upwind displacement dx", "m per step; <0 = upwind",
           (-0.06, 0.06), 40),
          ("dy", "lateral displacement dy", "m per step", (-0.06, 0.06), 40),
          ("turn", "turn action", "[0, 1]; 0.5 = straight", (0, 1), 30),
          ("move", "move action", "[0, 1]", (0, 1), 30),
          ("stray_distance", "stray distance", "m from plume", (0, 2), 40)]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, (key, title, unit, rng, nbins) in zip(axes.ravel(), PANELS):
    bins = np.linspace(rng[0], rng[1], nbins + 1)
    ctr = 0.5 * (bins[:-1] + bins[1:])
    for r, c in REGIME_COLORS.items():
        v = samples[r][key]
        if not len(v):
            continue
        h, _ = np.histogram(v, bins=bins, density=True)
        ax.plot(ctr, h, color=c, lw=1.8, label=r)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(unit, fontsize=9)
    ax.set_ylabel("density", fontsize=9)
axes[0, 0].legend(fontsize=9)
fig.suptitle("The three regimes are distinct motor programs "
             "(constant plume, HOME/non-HOME balanced)")
fig.tight_layout()
savefig(fig, "regime_dists.png")

The distributions separate cleanly. TRACK points upwind (head direction
concentrated at +/-180 deg, dx peaked at the most negative bin), moves at
full speed (move action pinned at 1.0) and keeps the stray distance pinned
near zero. RECOVER sits in between on every axis. SEARCH restructures the
whole motor pattern: the head direction shifts off the upwind pole, the move
action drops to a slower mode (~0.75), a sharp lateral-displacement
preference appears in dy (the directed casting of the paper), and the stray
distance spreads across the full 0-2 m range instead of hugging the plume.
This is the signature of a distinct search program, not of "tracking done
badly".

## 9. Agents track the plume centerline, not the wind

The paper's key behavioral discovery, and the one with a direct experimental
prediction for insects. On `switch45` the wind rotates +45 deg once,
mid-episode, and the plume downstream of the switch keeps the old direction
while the plume near the source follows the new one, leaving a bent
centerline. A pure wind-following agent ($a_t = f(\text{current wind})$)
would turn 45 deg as soon as the wind turns. A plume-shape-following agent
($a_t = f(\text{local plume geometry, odor history, wind history})$) should
instead stay anti-parallel to the **local plume centerline**.

To quantify this, the course direction (the direction of the agent's actual
displacement, in the ground frame) is compared against two references: the
current wind, and the local centerline tangent (median tangent of centerline
points within 0.02 m in x of the agent). The histograms below use TRACK-regime
timesteps only, with pre-switch timesteps on switch45 trimmed (wind within
15 deg of +x), following the reference. Angle convention: 0 = parallel to the
reference, +/-180 = anti-parallel, i.e. pointing back up the plume, where a
tracker should sit.

In the paper, the two references dissociate on switch45 exactly as the
centerline hypothesis predicts: course-vs-wind shifts by about 45 deg while
course-vs-centerline stays at +/-180, meaning the agents read the plume's
local geometry rather than the instantaneous wind. Our top seed only
partially reproduces this. The control condition behaves as expected (both
histograms peak at +/-180), and on switch45 the two references do dissociate,
but in the opposite direction from the paper: course-vs-wind stays at +/-180
while course-vs-centerline shifts to about +110 deg. This agent therefore
leans more on the current wind and less on the plume's large-scale shape than
the paper's agents do. With 5 seeds instead of 14 we cannot tell how
representative this is; the analysis itself (fixed grid, TRACK-only
timesteps, centerline tangent matching) is identical to the reference, so the
notebook remains a direct test bed for the paper's prediction.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
bins = np.linspace(0, 1, 37)
ctr = 0.5 * (bins[:-1] + bins[1:])
for row, d in enumerate(["constantx5b5", "switch45x5b5"]):
    eps_d = load_top(d)
    plume = load_or_generate_plume(d, PLUME_DATA_DIR)
    trim = 15.0 if "switch" in d else None
    rel_wind, rel_cl = [], []
    for ep in eps_d:
        fr = episode_to_frame(ep)
        cd = course_direction_frame(ep, plume, trim_wind_deg=trim)
        m = cd["mask"] & (fr["regime"][:len(cd["mask"])] == "TRACK")
        rel_wind.append(cd["cd_rel_wind_u"][m])
        rel_cl.append(cd["cd_rel_centerline_u"][m])
    for col, (vals, ref) in enumerate([(rel_wind, "wind"),
                                       (rel_cl, "centerline")]):
        ax = axes[row, col]
        v = np.concatenate(vals)
        h, _ = np.histogram(v, bins=bins, density=True)
        ax.plot(ctr, h, color=REGIME_COLORS["TRACK"], lw=1.8)
        ax.axvline(0.5, color="k", ls="--", lw=0.8)
        ax.set_title(f"{d}: TRACK course direction vs {ref} "
                     f"(n={len(v)} steps)", fontsize=10)
        if row == 1:
            ax.set_xticks(ANGLE_TICKS, ANGLE_TICKLABELS)
            ax.set_xlabel("relative angle (deg)")
axes[0, 0].set_ylabel("density")
axes[1, 0].set_ylabel("density")
fig.suptitle("Agents track the plume centerline, not the current wind "
             "(TRACK regime)")
fig.tight_layout()
savefig(fig, "course_direction.png")

## 10. What the RNN remembers

The second core result: the hidden state encodes **four task-relevant memory
variables** that were never given as inputs. Linear decoding from the hidden
states recovers, besides the instantaneous observations themselves:

1. **head direction** in the global frame (the observation only contains wind
   in the egocentric frame; the paper compares this to ring-attractor head
   direction representations in the insect central complex);
2. **$T_{last}$**, the time since the last odor encounter, the implicit clock
   behind the regime switches of section 8;
3. **odor EWMA**, an exponentially weighted average of concentration over a
   short window (~7 steps, 0.3 s; the paper decodes it with R² ~ 0.9);
4. **odor encounter rate**, an EWMA of odor onsets over a much longer window
   (~47 steps, 1.9 s; R² ~ 0.6 in the paper).

The two odor memories are different variables: a single strong whiff gives
high concentration but low encounter rate, while a train of weak whiffs does
the opposite, and they carry different information about whether the agent
still "has" the plume. The two-timescale signature shows up directly when the
decoding window is scanned (first plot): the concentration decode peaks at a
short window, the encounter decode at a long one. Our top seed reproduces
both the separation and the ranking: the odor EWMA decodes best at a 12-step
window (R² = 0.84, paper: ~7 steps at R² ~ 0.9) and the encounter rate at a
24-step window (R² = 0.40, paper: ~47 steps at R² ~ 0.6), while head
direction (sin/cos R² = 0.93/0.68) and T_last (R² = 0.56) are also clearly
readable.

To show these variables are behaviorally meaningful and not just visible in
PCA, the paper trains a random-forest readout of the *discretized actions*
from (a) the instantaneous observations alone and (b) the observations plus
the four memory variables. In the paper this adds 10-18 points of
cross-validated accuracy over the instantaneous inputs, and 26-51 points over
the majority-class baseline; $T_{last}$ is often the most important feature.
Our top seed reproduces the pattern on the turn channel (3-class accuracy
0.50 -> 0.66, +16 points). The move channel gains nothing, but for an
instructive reason: move is already decoded at ~1.0 accuracy from the
instantaneous observations alone, because the policy maps current wind and
odor almost directly onto forward speed. It is the turning decision, where
the history matters, that the memory variables unlock.

In [ ]:
eps_home = [e for e in eps if e["outcome"] == "HOME"]
scan_eps = eps_home if len(eps_home) >= 20 else eps
scan = window_scan_r2(scan_eps)
w_sec = scan["windows"] * 0.04
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(w_sec, scan["r2_ewm"], "o-", label="odor EWMA (concentration)")
ax.plot(w_sec, scan["r2_enc"], "s-", label="odor encounter rate")
ax.axvline(scan["best_ewm"] * 0.04, color="tab:blue", ls="--", lw=0.8)
ax.axvline(scan["best_enc"] * 0.04, color="tab:orange", ls="--", lw=0.8)
ax.set_xlabel("window length (s)")
ax.set_ylabel("R$^2$ (linear decode from hidden states)")
ax.set_title(f"Integration-time window scan (best: EWMA {scan['best_ewm']} "
             f"steps R$^2$={scan['r2_ewm'].max():.2f}; ENC "
             f"{scan['best_enc']} steps R$^2$={scan['r2_enc'].max():.2f})")
ax.legend()
fig.tight_layout()
savefig(fig, "memory_window_scan.png")

In [ ]:
# Linear decoding of task variables (incl. the memory variables at their best windows)
best = {"ewm": scan["best_ewm"], "enc": scan["best_enc"]}
COLS = ["wind_x", "wind_y", "odor", "head_cos", "head_sin", "T_last",
        "odor_ewm", "odor_enc"]
H, Ys = [], {c: [] for c in COLS}
dec_eps = (eps_home if len(eps_home) >= 20 else eps)[:80]
for ep in dec_eps:
    fr = episode_to_frame(ep)
    feats = extended_odor_features(fr["odor_obs"], [best["ewm"], best["enc"]])
    ang = np.asarray(ep["agent_angle"], dtype=np.float64)[:len(fr["loc_x"])]
    H.append(np.asarray(ep["states"], dtype=np.float64))
    Ys["wind_x"].append(fr["wind_x_obs"])
    Ys["wind_y"].append(fr["wind_y_obs"])
    Ys["odor"].append(fr["odor_clip"])
    Ys["head_cos"].append(np.cos(ang))
    Ys["head_sin"].append(np.sin(ang))
    Ys["T_last"].append(np.minimum(fr["odor_lastenc"], 50) * 0.04)
    Ys["odor_ewm"].append(feats[f"odor_ewm_{best['ewm']}"])
    Ys["odor_enc"].append(feats[f"odor_enc_{best['enc']}"])
H = np.vstack(H)
Y = np.stack([np.concatenate(Ys[c]) for c in COLS], axis=1)
r2 = behavior_regression_r2(H, Y)
NICE = {"wind_x": "wind x (egocentric, input)", "wind_y": "wind y (egocentric, input)",
        "odor": "odor (instantaneous, input)", "head_cos": "head direction cos (memory 1)",
        "head_sin": "head direction sin (memory 1)", "T_last": "time since last odor (memory 2)",
        "odor_ewm": f"odor EWMA, {best['ewm']}-step window (memory 3)",
        "odor_enc": f"odor encounter rate, {best['enc']}-step window (memory 4)"}
fig, ax = plt.subplots(figsize=(8, 4.2))
order = np.argsort(r2)
ax.barh(np.arange(len(COLS)), r2[order], color="steelblue")
for k, j in enumerate(order):
    ax.text(r2[j] + 0.01, k, f"{r2[j]:.2f}", va="center", fontsize=9)
ax.set_yticks(np.arange(len(COLS)), [NICE[COLS[j]] for j in order], fontsize=9)
ax.set_xlim(0, 1.05)
ax.set_xlabel("R$^2$ (linear decode from 64-d hidden state)")
ax.set_title("The four memory variables are linearly readable from the hidden state")
fig.tight_layout()
savefig(fig, "decoding_r2.png")
print("decode R^2:", {NICE[c]: round(float(r2[k]), 2) for k, c in enumerate(COLS)})

In [ ]:
# RandomForest action prediction: instantaneous vs +memory features
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def discretize(v, edges):
    return np.clip(np.digitize(v, edges), 0, len(edges))


X_inst, X_mem, Y_turn, Y_step = [], [], [], []
for ep in dec_eps:
    fr = episode_to_frame(ep)
    feats = extended_odor_features(fr["odor_obs"], [best["ewm"], best["enc"]])
    ang = np.asarray(ep["agent_angle"], dtype=np.float64)[:len(fr["loc_x"])]
    inst = np.stack([fr["wind_x_obs"], fr["wind_y_obs"], fr["odor_clip"]],
                    axis=1)
    mem = np.concatenate([inst,
                          np.stack([np.cos(ang), np.sin(ang),
                                    np.minimum(fr["odor_lastenc"], 50) * 0.04,
                                    feats[f"odor_ewm_{best['ewm']}"],
                                    feats[f"odor_enc_{best['enc']}"]],
                                   axis=1)], axis=1)
    X_inst.append(inst); X_mem.append(mem)
    Y_turn.append(discretize(fr["turn"], [1 / 3, 2 / 3]))
    Y_step.append(discretize(fr["step"], [0.5]))
X_inst, X_mem = np.vstack(X_inst), np.vstack(X_mem)
Y_turn, Y_step = np.concatenate(Y_turn), np.concatenate(Y_step)

rf_acc = {}
for name, X, Y in [("turn (3-class)", X_inst, Y_turn),
                   ("turn (3-class)", X_mem, Y_turn),
                   ("move (2-class)", X_inst, Y_step),
                   ("move (2-class)", X_mem, Y_step)]:
    clf = RandomForestClassifier(n_estimators=30, n_jobs=-1, random_state=137)
    acc = cross_val_score(clf, X, Y, cv=3).mean()
    rf_acc.setdefault(name, []).append(acc)
    print(f"[rf] {name} {'+memory' if X is X_mem else 'instant'}: {acc:.3f}")
majority = {"turn (3-class)": np.bincount(Y_turn).max() / len(Y_turn),
            "move (2-class)": np.bincount(Y_step).max() / len(Y_step)}

fig, ax = plt.subplots(figsize=(6.5, 4))
xs = np.arange(2)
w = 0.35
for i, (label, j) in enumerate([("instantaneous only", 0),
                                ("+ memory variables", 1)]):
    ax.bar(xs + (i - 0.5) * w,
           [rf_acc["turn (3-class)"][j], rf_acc["move (2-class)"][j]],
           w, label=label)
for i, name in enumerate(["turn (3-class)", "move (2-class)"]):
    ax.axhline(majority[name], color=f"C{i}", ls=":", lw=1)
ax.set_xticks(xs, ["turn (3-class)", "move (2-class)"])
ax.set_ylabel("cross-validated accuracy")
ax.set_title("Action prediction: memory variables matter "
             "(dotted = majority class)")
ax.legend()
fig.tight_layout()
savefig(fig, "rf_classifier.png")

## 11. Common subspace and limit-cycle search dynamics

The neural-dynamics core of the paper. One PCA is fit on the stacked hidden
states of all 240 evaluation episodes (the "common subspace"). Two facts
emerge. First, the activity is **low-dimensional**: 5-8 principal components
explain about 90% of the variance, out of 64 units. Second, the state space
organizes into **two dynamical regimes**, matching the behavioral regimes of
section 8:

* TRACK lives on a **funnel-like manifold**: trajectories entering from
  different histories are compressed onto a similar path, which is why the
  agent can snap back into stable tracking as soon as it re-encounters odor;
* SEARCH is a **quasi-limit-cycle**: with the plume lost, the motor program
  is a rhythmic cast-drift-cast oscillation, and the hidden state traces a
  closed loop with a period of ~17-28 steps (0.7-1.1 s) in 4 of the paper's
  5 agents (one agent showed no clear periodicity).

RECOVER occupies an amorphous transition region between the two, so the right
picture is two regimes plus a transition region, not three discrete states.

One honest deviation shows up in the period histogram below: our agents do
not develop the paper's long-period search oscillation. The median SEARCH
period is 2 steps (against the paper's 17-28), i.e. the lost-plume state
jitters rather than tracing a slow loop. The most plausible reading is that
the limit cycle is a property of the well-trained search program: the paper
selects the 5 best of 14 seeds, while our weaker seeds search poorly and even
the top seed's casting is less stereotyped. The low-dimensionality result, by
contrast, reproduces cleanly (90% of variance in 6 PCs for the top seed).

The first plot shows the scree curve, episodes colored by regime in PC space,
and the odor-triggered average of the first PCs. The second measures the
SEARCH periodicity directly: for each SEARCH-dominated episode, the period is
the lag $p$ that minimizes the distance between $h_t$ and $h_{t-p}$ over the
final 50 steps (the reference's signed-sum measure), and the histogram
collects periods across episodes and datasets.

In [ ]:
pca = fit_common_pca(eps, n_components=12)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ax = axes[0]
k = len(pca.explained_variance_ratio)
cum = np.cumsum(pca.explained_variance_ratio)
ax.plot(np.arange(1, k + 1), cum, "o-")
n90 = int(np.searchsorted(cum, 0.9)) + 1
ax.axhline(0.9, color="k", ls="--", lw=0.8)
ax.set_xlabel("number of PCs")
ax.set_ylabel("cumulative explained variance")
ax.set_title(f"common subspace scree (90% at {n90} PCs)")
ax = axes[1]
for ep in eps[:12]:
    fr = episode_to_frame(ep)
    Z = transform_states(pca, ep["states"])
    for r, c in REGIME_COLORS.items():
        m = fr["regime"] == r
        ax.scatter(Z[m, 0], Z[m, 1], s=3, c=c, alpha=0.6)
for r, c in REGIME_COLORS.items():
    ax.scatter([], [], c=c, label=r)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("trajectories colored by regime")
ax.legend(fontsize=8)
ax = axes[2]
ots = odor_triggered_states(eps_home[:80] if eps_home else eps,
                            pre=10, post=25)
if len(ots):
    Zo = transform_states(pca, ots.reshape(-1, ots.shape[-1]))
    Zo = Zo.reshape(ots.shape[0], ots.shape[1], -1)
    tt = (np.arange(ots.shape[1]) - 10) * 0.04
    for pc in range(3):
        ax.plot(tt, Zo[:, :, pc].mean(0), label=f"PC{pc + 1}")
    ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel("t from odor onset (s)")
ax.set_title(f"odor-triggered PCs (n={len(ots)})")
ax.legend(fontsize=8)
fig.suptitle("Common PCA subspace (constantx5b5, top seed)")
fig.tight_layout()
savefig(fig, "common_subspace.png")

In [ ]:
# Limit-cycle periodicity of the SEARCH regime (reference get_period,
# report_common_subspace: signed-sum-over-units distance between h_t and
# h_{t-p}, scored on the last 50 steps of SEARCH-dominated episodes)
def _search_tail_frac(ep, tail=50):
    seg = np.asarray(episode_to_frame(ep)["regime"])[-tail:]
    return float((seg == "SEARCH").mean())

eps_pooled = list(eps)
for _ds in ("switch45x5b5", "noisy3x5b5"):
    try:
        eps_pooled += load_top(_ds)
    except FileNotFoundError:
        pass
search_eps = [e for e in eps_pooled
              if e["length"] >= 100 and _search_tail_frac(e) >= 0.8]
periods, periodicity = [], []
for ep in search_eps:
    p, dists = limit_cycle_period(ep["states"], max_period=50, tail=50,
                                  measure="signed_sum")
    periods.append(p)
    periodicity.append(dists[p - 1] / (np.median(dists) + 1e-12))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.hist(periods, bins=np.arange(0.5, 51.5, 1), color="steelblue")
med_p = float(np.median(periods)) if periods else float("nan")
ax.axvline(med_p, color="k", ls="--", lw=0.8)
ax.set_xlabel("limit-cycle period (steps)")
ax.set_ylabel("episodes")
ax.set_title(f"SEARCH periodicity (n={len(periods)}, median {med_p:.0f} steps)\n"
             f"paper: 17-28 steps in 4/5 agents", fontsize=10)
ax = axes[1]
if search_eps:
    ep = search_eps[int(np.argmin(periodicity))]
    p_ep = periods[int(np.argmin(periodicity))]
    Z = transform_states(pca, ep["states"])
    fr = episode_to_frame(ep)
    m = fr["regime"] == "SEARCH"
    ax.plot(Z[:, 0], Z[:, 1], "-", c="0.7", lw=0.8)
    ax.plot(Z[m, 0], Z[m, 1], ".", c=REGIME_COLORS["SEARCH"], ms=4)
    ax.plot(Z[-50:, 0], Z[-50:, 1], "-", c="k", lw=1.5,
            label="last 50 steps")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_title(f"most periodic SEARCH episode (period {p_ep} steps)")
    ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "limit_cycle.png")

## 12. Eigenspectrum and integration timescales

What did training do to the recurrent weights? For a vanilla tanh RNN the
linearized dynamics around $h = 0$ are governed by the eigenvalues of
$W_{hh}$. With the paper's $N(0, 1/\sqrt{64})$ init the circular law puts
the spectral radius at $\sigma\sqrt{n} = 1$: the untrained network sits at
the edge of stability, with a handful of very long-lived modes (the pretrain
timescales in the right panel reach past 100 steps). After training, the
paper finds several **unstable modes**
with $|\lambda| > 1$ in every trained agent, typically including at least
one strictly real eigenvalue. Our five seeds show 2-6 unstable modes each;
a strictly real unstable mode appears in three of them (the other two have
only complex-conjugate unstable pairs), a quantitative difference from the
paper that is expected at our smaller seed count. These modes do not blow up because the tanh saturation and the
sensory input hold them in check; the paper interprets them as the substrate
of the state-dependent memory dynamics (and is explicit that a complete
algorithmic account is still missing).

The same spectrum gives the network's **integration timescales**,
$\tau_i = 1/|\ln|\lambda_i||$ in units of steps: how long each linear mode
remembers its input. The picture that emerges for the trained network is "fast
dynamics plus a few slow memory modes": almost all modes integrate over <= 12
steps (0.5 s, so most of plume-tracking computation is short-timescale), with
one or two much slower modes (the paper reports top timescales such as 56.5,
13.0, 7.7, 6.8, 5.8 steps; our seeds' median top-5 are 28.9, 27.5, 16.0,
13.5, 6.7 steps). Note what training does to the bulk of the spectrum: it
*shortens* most modes relative to the edge-of-stability initialization, while
pushing a few modes out past the unit circle. The trajectory-dependent version recomputes the spectrum from the
recurrence Jacobian $J(t) = \mathrm{diag}(1 - h_t^2)\, W_{hh}$ along actual
evaluation episodes, showing the same separation at the network's real
operating points.

In [ ]:
agent = AutoModel.from_pretrained(agent_dir(TOP, 2))
print("per-seed unstable modes (|lambda| > 1):")
for seed in SEEDS:
    if not agent_exists(seed, 2):
        continue
    ev = eigenspectrum(AutoModel.from_pretrained(agent_dir(seed, 2)))
    r = np.abs(ev)
    print(f"  seed{seed}: {int((r > 1).sum())} unstable "
          f"({int(((r > 1) & (np.abs(ev.imag) < 1e-6)).sum())} real)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
th = np.linspace(0, 2 * np.pi, 100)
ax = axes[0]
ev_pre = eigenspectrum(AutoModel.from_pretrained(agent_dir(TOP, "pretrain")))
ev_post = eigenspectrum(agent)
ax.plot(np.cos(th), np.sin(th), "k--", lw=0.8)
ax.scatter(ev_pre.real, ev_pre.imag, s=14, alpha=0.7, label="pretrain")
ax.scatter(ev_post.real, ev_post.imag, s=14, alpha=0.7, label="trained")
ax.set_xlabel("Re"); ax.set_ylabel("Im"); ax.set_aspect("equal")
ax.set_title(f"W_hh eigenspectrum (seed {TOP})")
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(np.arange(1, 65), sorted_taus(ev_pre), "o-", ms=3, label="pretrain")
ax.plot(np.arange(1, 65), sorted_taus(ev_post), "s-", ms=3, label="trained")
ax.axhline(12, color="k", ls="--", lw=0.8)
ax.set_xlabel("mode (sorted)"); ax.set_ylabel("timescale (steps)")
ax.set_yscale("log")
ax.set_title("integration timescales of W_hh")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "eigenspectrum.png")

In [ ]:
# Trajectory-dependent timescales from recurrence Jacobians along episodes
curves = episode_timescale_curves(agent, eps[:8])
med = np.median(curves, axis=0)
q25, q75 = np.percentile(curves, [25, 75], axis=0)
fig, ax = plt.subplots(figsize=(7, 4.2))
xs = np.arange(1, curves.shape[1] + 1)
ax.fill_between(xs, q25, q75, alpha=0.3)
ax.plot(xs, med, "k-", lw=1.5, label="median over timesteps")
ax.axhline(12, color="r", ls="--", lw=0.8, label="12 steps (~0.5 s)")
ax.set_xlabel("mode (sorted)"); ax.set_ylabel("timescale (steps)")
ax.set_yscale("log")
ax.set_title("Stimulus-integration timescales along trajectories "
             "(recurrence Jacobians)")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "timescales.png")
print(f"top-5 median timescales: {np.round(med[:5], 1)} "
      f"(paper e.g. 56.5, 13.0, 7.7, 6.8, 5.8)")

## 13. Transition-time asymmetry (TTL vs TTT)

Entering and leaving the tracking regime are not symmetric processes. After
**re-encountering** the plume, the neural state snaps back into the tracking
region quickly (TTT, time-to-track). After **losing** the plume, it takes
longer to settle into the lost/search region (TTL, time-to-lost). The paper
finds TTL > TTT in 4 of 5 agents, and connects the asymmetry to the fly
literature, where cross-wind casting after plume loss takes about twice as
long as the upwind surge after re-encounter. In our run only one seed has
both enough HOME episodes and enough long OOB episodes to define the two
centroids (the weaker seeds almost never reach home, and the strongest rarely
strays for long), and that seed shows TTL > TTT, consistent with the paper. "Realizing the plume is gone"
is a slower neural process than "exploiting that it is back".

Operationally: the HOME and OOB **neural centroids** are the mean final
25-step activity of successful episodes and of long-OOB episodes, and the
transition times measure how quickly the state enters the corresponding
centroid's neighborhood (radius = half the inter-centroid distance). TTL runs
from the final plume exit of an OOB episode; TTT runs from an odor
re-encounter after at least 13 off-plume steps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
all_ttl, all_ttt, seed_med = [], [], []
for seed in SEEDS:
    try:
        eps_s = load_top("constantx5b5", seed=seed)
        cen = outcome_centroids(eps_s)
        tt = transition_times(eps_s, cen)
        all_ttl += tt["ttl"]; all_ttt += tt["ttt"]
        if tt["ttl"] and tt["ttt"]:
            seed_med.append((seed, np.median(tt["ttl"]), np.median(tt["ttt"])))
    except (FileNotFoundError, ValueError) as e:
        print(f"[ttc] seed{seed} skipped: {e}")
ax = axes[0]
bins = np.arange(0, 60, 2) * 0.04
ax.hist(np.array(all_ttt) * 0.04, bins=bins, alpha=0.6, density=True,
        label=f"TTT (re-enter -> track), median "
              f"{np.median(all_ttt) * 0.04:.2f} s")
ax.hist(np.array(all_ttl) * 0.04, bins=bins, alpha=0.6, density=True,
        label=f"TTL (exit -> lost), median "
              f"{np.median(all_ttl) * 0.04:.2f} s")
ax.set_xlabel("transition time (s)"); ax.set_ylabel("density")
ax.set_title("Neural-regime transition times")
ax.legend(fontsize=8)
ax = axes[1]
for seed, m_ttl, m_ttt in seed_med:
    ax.plot([0, 1], [m_ttt * 0.04, m_ttl * 0.04], "o-", label=f"seed{seed}")
ax.set_xticks([0, 1], ["TTT", "TTL"])
ax.set_ylabel("median transition time (s)")
ax.set_title("asymmetry per seed (paper: TTL > TTT in 4/5)")
ax.legend(fontsize=8)
fig.tight_layout()
savefig(fig, "transition_asymmetry.png")
n_up = sum(1 for _, a, b in seed_med if a > b)
print(f"TTL > TTT in {n_up}/{len(seed_med)} seeds")

## 14. Notes on faithfulness

1. **Scale**: 5 seeds (paper: 14, top-5 reported). PPO batch, step budget and
   all hyperparameters match the paper exactly (4 procs x 2048 steps = 8192;
   1M + 4M env steps per seed; rollout-global advantage normalization and
   VecNormalize obs/reward scaling as in the reference's ikostrikov-PPO + SB3
   stack).
2. Per-env independent RNG streams (framework convention).
3. The reference core init $N(0, 1/\sqrt{64})$, zero biases, is applied by
   `init_plume_agent_` rather than the family defaults.
4. The reference eval script adds +100 to any reward > 9 (a debug hack per
   the source comment); **not** ported.
5. evalCli's `seedmeta` per-seed regime thresholds are not ported (defaults
   RECOVER_MIN=12 / RECOVER_MAX=25 for all seeds).
6. The paper's GRU/MLP architecture controls (appendix) are not part of this
   notebook; the analysis suite is VRNN-only, as in the reference.
7. The random-forest action readout uses the same feature sets and 30-tree
   configuration as the reference, with 3-fold cross-validation.